In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from catboost import CatBoostRegressor

In [ ]:
data_train = pd.read_csv("./used-cars-price-prediction-19ds/train.csv")
data_test = pd.read_csv("./used-cars-price-prediction-19ds/test.csv")

In [ ]:
RANDOM_STATE = 12345
TRIM_LEVELS = {
    "ce": ["classic edition", "custom edition", "ce"], 
    "dx": ["deluxe", "dx"],
    "dl": ["deluxe level", "dl"],
    "ex": ["extra", "ex"], 
    "gl": ["grade level", "gl"],
    "gle": ["grade level extra", "gle"],
    "gt": ["grand touring", "gt"],
    "lx": ["luxury", "lx"],
    "le": ["luxury edition", "le"],
    "ls": ["luxury sport", "luxury special", "ls"],
    "lt": ["luxury touring", "lt"],
    "ltd": ["limited", "ltd"],
    "ltz": ["luxury touring special", "ltz"],
    "se": ["sport edition", "special edition", "special equipment", "se"],
    "sl": ["standart level", "standart", "sl"],
    "sle": ["standard level extra", "sle"],
    "slt": ["standard level touring", "slt"],
    "sv": ["Special Version", "sv"],
    "xlt": ["extra level touring", "xlt"],
    "base": ["base"],
    "e": ["edition", "e"], 
    "t": ["touring", "t"],
}

In [ ]:
def make_unique(data):
    if data == "ford truck":
        return "ford"
    if data == "gmc truck":
        return "gmc"
    if data == "land rover":
        return "landrover"
    if data == "mercedes-benz":
        return "mercedes"
    if data == "vw":
        return "volkswagen"
    if data == "dodge tk":
        return "dodge"
    if data == "mazda tk":
        return "mazda"
    else:
        return data

In [ ]:
def body_unique(data):
    if data.find("cab") != -1 or data == "koup":
        return "pick-up"
    if data.find("convertible") != -1:
        return "convertible"
    if data.find("coupe") != -1:
        return "coupe"
    if data.find("wagon") != -1:
        return "wagon"
    if data.find("van") != -1:
        return "van"
    if data.find("sedan") != -1:
        return "sedan"
    if data.find("van") != -1:
        return "van"
    else:
        return data

In [ ]:
def classify_trim_levels(trim_levels, data):
    for key in trim_levels:
        for index in trim_levels[key]:
            if index in data:
                return key
    return "unknown"

In [ ]:
def trim_unique(data):
    return classify_trim_levels(TRIM_LEVELS, data)

In [ ]:
def preprocessing(data):
    data = data.drop(["vin", "seller", "saledate"], axis=1)

    data["make"] = data["make"].apply(lambda x: str(x).lower().strip())
    data["model"] = data["model"].apply(lambda x: str(x).lower().strip())
    data["trim"] = data["trim"].apply(lambda x: str(x).lower().strip())
    data["body"] = data["body"].apply(lambda x: str(x).lower().strip())
    data["transmission"] = data["transmission"].apply(lambda x: str(x).lower().strip())
    data["state"] = data["state"].apply(lambda x: str(x).lower().strip())
    data["color"] = data["color"].apply(lambda x: str(x).lower().strip())
    data["interior"] = data["interior"].apply(lambda x: str(x).lower().strip())
    
    data["trim"] = data["trim"].apply(trim_unique)

    # mean_value_odometer = data["odometer"].mean()
    # mean_values_odometer = list(data.groupby("condition")["odometer"].mean().values)
    # mean_values_odometer_class = list(data.groupby("condition")["odometer"].mean().index)
    # index_odometer = mean_values_odometer.index(max(filter(lambda x: x < mean_value_odometer, mean_values_odometer), default=None))
    # class_value = mean_values_odometer_class[index_odometer]
    # data[(data["odometer"] > 0.19 * (10**6))] = mean_value_odometer
    # data[data["odometer"] == mean_value_odometer] = class_value

    return data

In [ ]:
def getfullitemsforOHE(catigories_list, catigories_columns):
    catigories_dict = {}
     
    for i in catigories_columns:
          catigories_dict[i] = []
          
    for i in catigories_dict:
         for g in catigories_list:
              if i in g:
                   catigories_dict[i].append(g)
    
    feated_list = []

    for i in catigories_dict:
         feated_list.append(catigories_dict[i])

    return feated_list

In [ ]:
data_train = preprocessing(data_train)
data_test = preprocessing(data_test)

In [ ]:
data_train.boxplot("sellingprice")

In [ ]:
full_data = pd.concat([data_train, data_test])
full_data = full_data.reset_index(drop=True)

In [ ]:
cat_columns = ["make", "model", "trim", "body", "transmission", "state", "color", "interior"]

In [ ]:
cats = getfullitemsforOHE(pd.get_dummies(full_data[cat_columns], drop_first=True).columns, cat_columns)

In [ ]:
ohe=OneHotEncoder(categories=cats, sparse=False, handle_unknown="ignore")

X_train = ohe.fit_transform(data_train[cat_columns])
ohe_train = pd.DataFrame(X_train,columns=ohe.get_feature_names_out(cat_columns))

X_test = ohe.fit_transform(data_test[cat_columns])
ohe_test = pd.DataFrame(X_test,columns=ohe.get_feature_names_out(cat_columns))

In [ ]:
data_train = data_train.drop(cat_columns, axis=1)
data_train = data_train.join(ohe_train)

data_test = data_test.drop(cat_columns, axis=1)
data_test = data_test.join(ohe_test)

In [ ]:
features = data_train.drop(["sellingprice"], axis=1)
target = data_train["sellingprice"]

In [ ]:
cat = CatBoostRegressor(n_estimators=2000, depth=16, random_state=RANDOM_STATE, silent=True, task_type="GPU", devices="0:1")

In [ ]:
cat.fit(features, target, plot=True)

In [ ]:
results = pd.DataFrame([cat.predict(data_train),  pd.read_csv("./used-cars-price-prediction-19ds/test.csv")["vin"]]).to_csv("results.csv", index=False)